# Installs if needed

In [14]:
# !pip install glob
# !pip install pandas
# !pip install numpy
# !pip install matplotlib
# !pip install seaborn
# !pip install fastparquet
# !pip install sklearn

In [32]:
# imports
import pandas as pd
import numpy as np
import glob
import pyarrow

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Data Engineering Pipeline 
### 3. Extract, Transform, Load (ETL)
---
Extracting data from all the files. Combine everything into one dataframe.

All the dataframes that were created from the CSV, JSON, and Parquet files. They will now be combined and be ready to be cleaned.

In [16]:
# Load all CSV Files
csv_files = glob.glob('final_project_data_sp2026_L/*.csv')
df_csv = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

# Load all JSON Files
json_files = glob.glob('final_project_data_sp2026_L/*.json')
df_json = pd.concat((pd.read_json(f, lines=True, orient='records') for f in json_files), ignore_index=True)

# Load Parquet Files
parquet_files = glob.glob('final_project_data_sp2026_L/*.parquet')
df_parquet = pd.concat((pd.read_parquet(f, engine='fastparquet') for f in parquet_files), ignore_index=True)

# check all dataframe shapes
print("CSV DataFrame:")
print(df_csv.shape)

print("\nJSON DataFrame:")
print(df_json.shape)

print("\nParquet DataFrame:")
print(df_parquet.shape)    


CSV DataFrame:
(4004, 79)

JSON DataFrame:
(27805, 79)

Parquet DataFrame:
(31320, 79)


In [17]:
"""
Combine all the data into a single DataFrame for analysis and modeling. This will allow us to clean up the data. 
"""
total_df = pd.concat([df_csv, df_json, df_parquet], ignore_index=True)

# check the combined dataframe
print("\nCombined DataFrame:")
print(total_df.shape[0], "rows,", total_df.shape[1], "columns")



Combined DataFrame:
63129 rows, 79 columns


### 4. Data Transformation 
---
...

In [18]:
"""
Remove all the duplicates that are in the combined dataframe. 
"""

print("\nBefore duplicate removal:", total_df.shape)
total_df = total_df.drop_duplicates()
print("After duplicate removal:", total_df.shape)



Before duplicate removal: (63129, 79)
After duplicate removal: (49366, 79)


In [ ]:
"""
Handle all the missing values in the combined dataframe. 
"""
# Check for missing values
print("\nMissing values in each column:", total_df.isnull().sum().sort_values(ascending=False))

# fill in the numerical  columns with median values
num_cols = total_df.select_dtypes(include=[np.number]).columns
total_df[num_cols] = total_df[num_cols].fillna(total_df[num_cols].median())

# fill categorical columns with mode values
cat_cols = total_df.select_dtypes(include=['object']).columns
for col in cat_cols:
    total_df[col] = total_df[col].fillna(total_df[col].mode()[0])


Missing values in each column: Flow Bytes/s            4
 Flow Packets/s         2
 Destination Port       0
 Average Packet Size    0
 Fwd Avg Bulk Rate      0
                       ..
 Bwd IAT Std            0
 Bwd IAT Mean           0
Bwd IAT Total           0
 Fwd IAT Min            0
 Label                  0
Length: 79, dtype: int64


In [21]:
"""
Remove outliers from the combined dataframe using IQR method
"""
def iqr_outlier_removal(df, column):
    for col in column: 
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df

total_df = iqr_outlier_removal(total_df, num_cols)

In [22]:
"""
Feature Scaling
"""
scaler = StandardScaler()
total_df[num_cols] = scaler.fit_transform(total_df[num_cols])   

# 5. Data Storage 
We used a SQLite DB to store all the data

In [33]:
import sqlite3

sqlconnection = sqlite3.connect('final_project_data.db')

total_df.to_sql('final_project_data', sqlconnection, if_exists='replace', index=False)

query = 'SELECT COUNT(*) FROM final_project_data'

sqlconnection.close()


# 6. Reading Data

Reading data from the SQLite3 Database.

In [41]:
# connect to database
sqlconnection = sqlite3.connect('final_project_data.db')

# read the table into a pandas dataframe
final_df = pd.read_sql_query('SELECT * FROM final_project_data', sqlconnection)

# check if the dataframe is good
print('Loaded df shape:', final_df.shape)
print(final_df.head())

# close the connection 
sqlconnection.close()

Loaded df shape: (4983, 79)
    Destination Port   Flow Duration   Total Fwd Packets  \
0                0.0        0.578791           -1.377984   
1                0.0        0.674139           -0.376599   
2                0.0       -1.256471           -1.377984   
3                0.0       -1.497268           -1.377984   
4                0.0        0.842117           -1.377984   

    Total Backward Packets  Total Length of Fwd Packets  \
0                      0.0                    -1.435442   
1                      0.0                     1.063947   
2                      0.0                    -0.305581   
3                      0.0                    -2.120206   
4                      0.0                     0.276469   

    Total Length of Bwd Packets   Fwd Packet Length Max  \
0                           0.0               -1.381513   
1                           0.0                0.924411   
2                           0.0               -0.245759   
3                   

# 7. Exploratory Data Analysis
### 8 Step Process
Exploratory data analysis (EDA) is essential for understanding the dataset and making informed decisions about model selection and feature engineering. EDA may include the following: 

---
1. Identifying the shape of the dataset
2. Unify the columns/features names
3. Identifiying the unique values in the class variable 
4. Identifying if the dataset has missing data
5. Identifiying columns with a high percentage of missing data
6. Performing univariate analysis
    - stats of columns: mean, SD, or variance
    - identify columns with low or near-zero variance. These features/columns 
    contain little information and may be removed during feature engineering.
    - Visualization: bar plots, pie charts, boxplots, or violin plots
7. Performing bivariate analysis
    - generating pari plots of each pair of columns 
    - computing and visualizing the correlation matrix of the dataset columns or partial subset of them
8. Idenfity Outliers
    - Identify outliers using interquartile ratio (IQR): IQR = Q3 – Q1, with data with values less than $(Q1 – 1.5*IQR)$ and larger than $(Q3 + 1.5*IQR)$ are considered outliers.
    - Identifying outliers using histograms or boxplots 

In [50]:
# 1. Identifying the shapte of the dataset. Check the shape and columns and the information about the dataset.
print("\nDataset Shape:", final_df.shape)
print("\nDataset Columns:", final_df.columns)
print("\nDataset Info:", final_df.info())


Dataset Shape: (4983, 79)

Dataset Columns: Index(['destination_port', 'flow_duration', 'total_fwd_packets',
       'total_backward_packets', 'total_length_of_fwd_packets',
       'total_length_of_bwd_packets', 'fwd_packet_length_max',
       'fwd_packet_length_min', 'fwd_packet_length_mean',
       'fwd_packet_length_std', 'bwd_packet_length_max',
       'bwd_packet_length_min', 'bwd_packet_length_mean',
       'bwd_packet_length_std', 'flow_bytes/s', 'flow_packets/s',
       'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max',
       'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std',
       'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags',
       'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length',
       'bwd_header_length', 'fwd_packets/s', 'bwd_packets/s',
       'min_packet_length', 'max_packet_length', 'packet_length_mean',
       'packet_length_std', 'packet_length_var

In [51]:
# 2. Unify the columns/features names. Clean column names.
final_df.columns = final_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

In [52]:
# 3. Identifiying the unique values in the class label variable. Find the unique values in the target variable. 


In [53]:
# 4. Identifying if the dataset has missing data. Check for missing values in the dataset and handle them appropriately (e.g., imputation, removal). 
print(final_df.isnull().sum())

destination_port               0
flow_duration                  0
total_fwd_packets              0
total_backward_packets         0
total_length_of_fwd_packets    0
                              ..
idle_mean                      0
idle_std                       0
idle_max                       0
idle_min                       0
label                          0
Length: 79, dtype: int64


In [ ]:
# 5. Univariate analysis